# SerpApi live flight and hotel snapshots

This notebook queries Google Flights and Google Hotels through SerpApi for upcoming travel dates. It records the collection timestamp and writes CSV snapshots into the project `data/` directory.

**This is not a historical archive:** it cannot retrieve prices from past years. Do not use these rows as multi-year historical training data.

Set `SERPAPI_API_KEY` in your environment before running the request cells. The key is read from the environment and is never stored in this notebook.

In [ ]:
import os, time
import requests
import pandas as pd
import datetime as dt
from pathlib import Path
from datetime import timezone
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_KEY = os.getenv("SERPAPI_API_KEY")
if not API_KEY:
    raise RuntimeError("Set SERPAPI_API_KEY in your environment before running collection")

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").exists())
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
dests = ["DXB", "JED", "RUH", "DOH", "KWI", "AMM", "IST", "ATH", "FCO", "CDG", "FRA", "LHR"]
dates = [(dt.date.today() + dt.timedelta(days=d)).isoformat() for d in (14, 30, 60)]
collected_at = dt.datetime.now(timezone.utc).isoformat(timespec="seconds")
rows = []

s = requests.Session()
s.mount("https://", HTTPAdapter(max_retries=Retry(total=4, backoff_factor=3, status_forcelist=[429, 500, 502, 503, 504])))

for dest in dests:
    for d in dates:
        try:
            data = s.get("https://serpapi.com/search", params={
                "engine": "google_flights", "departure_id": "CAI", "arrival_id": dest,
                "outbound_date": d, "type": 2, "currency": "EGP", "gl": "eg", "hl": "en",
                "api_key": API_KEY,
            }, timeout=90).json()
        except (requests.exceptions.RequestException, ValueError) as e:
            print("skipped", dest, d, type(e).__name__)
            continue
        if "error" in data:
            print("api error", dest, d, data["error"])
            continue
        for f in data.get("best_flights", []) + data.get("other_flights", []):
            seg = f["flights"]
            rows.append({
                "collected_at": collected_at,
                "origin": "CAI",
                "destination": dest,
                "destination_city": data["airports"][0]["arrival"][0]["city"],
                "destination_country": data["airports"][0]["arrival"][0]["country"],
                "flight_date": d,
                "airline": ", ".join(dict.fromkeys(x["airline"] for x in seg)),
                "flight_number": ", ".join(x["flight_number"] for x in seg),
                "departure_time": seg[0]["departure_airport"]["time"],
                "arrival_time": seg[-1]["arrival_airport"]["time"],
                "duration_min": f["total_duration"],
                "stops": len(seg) - 1,
                "layovers": ", ".join(l["id"] for l in f.get("layovers", [])),
                "price_egp": f.get("price"),
                "airplane": seg[0].get("airplane"),
                "travel_class": seg[0].get("travel_class"),
                "co2_g": f.get("carbon_emissions", {}).get("this_flight"),
                "price_level": data.get("price_insights", {}).get("price_level"),
            })
        time.sleep(2)

df = pd.DataFrame(rows)
df["departure_time"] = pd.to_datetime(df["departure_time"])
df["arrival_time"] = pd.to_datetime(df["arrival_time"])
flight_csv = DATA_DIR / "cai_international_flights.csv"
df.to_csv(flight_csv, mode="a", index=False, header=not flight_csv.exists())
print(df.shape)
df.head(10)

### Hotel data

In [ ]:
cities = {"DXB": "Dubai", "JED": "Jeddah", "RUH": "Riyadh", "DOH": "Doha", "KWI": "Kuwait City", "AMM": "Amman",
          "IST": "Istanbul", "ATH": "Athens", "FCO": "Rome", "CDG": "Paris", "FRA": "Frankfurt", "LHR": "London"}
check_in = dates[1]
check_out = (dt.date.fromisoformat(check_in) + dt.timedelta(days=3)).isoformat()
hotel_rows = []

for dest, city in cities.items():
    try:
        data = s.get("https://serpapi.com/search", params={
            "engine": "google_hotels", "q": f"hotels in {city}",
            "check_in_date": check_in, "check_out_date": check_out,
            "adults": 1, "currency": "EGP", "gl": "eg", "hl": "en",
            "api_key": API_KEY,
        }, timeout=90).json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print("skipped", dest, type(e).__name__)
        continue
    if "error" in data:
        print("api error", dest, data["error"])
        continue
    for p in data.get("properties", []):
        hotel_rows.append({
            "collected_at": collected_at,
            "destination": dest,
            "city": city,
            "check_in": check_in,
            "check_out": check_out,
            "name": p.get("name"),
            "type": p.get("type"),
            "hotel_class": p.get("extracted_hotel_class"),
            "rating": p.get("overall_rating"),
            "reviews": p.get("reviews"),
            "price_per_night_egp": p.get("rate_per_night", {}).get("extracted_lowest"),
            "total_price_egp": p.get("total_rate", {}).get("extracted_lowest"),
            "latitude": p.get("gps_coordinates", {}).get("latitude"),
            "longitude": p.get("gps_coordinates", {}).get("longitude"),
            "free_cancellation": p.get("free_cancellation"),
            "amenities": ", ".join(p.get("amenities", [])),
        })
    time.sleep(2)

hotels = pd.DataFrame(hotel_rows)
hotel_csv = DATA_DIR / "destination_hotels.csv"
hotels.to_csv(hotel_csv, mode="a", index=False, header=not hotel_csv.exists())
print(hotels.shape)
hotels.head(10)